In [ ]:
# Desactivamos wandb
import os
os.environ["WANDB_DISABLED"] = "true"

In [ ]:
# Comprobación de Google Colab
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

In [ ]:
import random
import numpy as np
import pandas as pd
import sklearn
import torch
import matplotlib

from pathlib import Path

if IN_COLAB:
    from abc import ABC, abstractmethod
    import time
    import json
    import shutil
    import kagglehub
    from typing import Callable
    import pickle
    import re
    import matplotlib.pyplot as plt
    from sklearn.model_selection import train_test_split
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.linear_model import SGDClassifier
    from sklearn.utils import resample
    from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, ConfusionMatrixDisplay
    from scipy.sparse import csr_matrix, save_npz, load_npz
    from datasets import Dataset
    from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments

    # src.utils
    class LyricsDatasetConfig:
        def __init__(self, config_path: str | Path):
            self.config_path = Path(config_path)

            try:
                with open(self.config_path, 'r') as f:
                    config_object = json.load(f)
                self.name = config_object["name"]
                self.csv_name = config_object["csv_name"]
                self.kagglehub_handle = config_object["kagglehub_handle"]
                self.csv_path = config_object["csv_path"] if config_object["csv_path"] else None
                self.cols_map = config_object["cols_map"]
                self.language_col = config_object["language_col"] if config_object["language_col"] else None
                self.target_language = config_object["target_language"] if config_object["target_language"] else None
                self.vectorizer_config = config_object["vectorizer_config"]

                if self.language_col and not self.target_language:
                    raise RuntimeError(
                        f"- Se ha especificado una columna de idioma pero no un idioma objetivo.")

                if not self.language_col and self.target_language:
                    raise RuntimeError(
                        f"- Se ha especificado un idioma objetivo pero no la columna de idioma.")

            except Exception as e:
                print(
                    f"- Error al cargar el fichero JSON con la configuración: {e}")

    class LyricsDatasetManager:
        """
        Clase dedicada a la obtención de los *datasets* utilizados
        """

        def __init__(
                self,
                directory: str | Path = "data"
        ):
            """
            Constructor de la clase DatasetManager.
            Args:
                dir_path (str | Path): directorio en que se almacenarán el/los *dataset/s*. Por defecto,
                                    se utiliza `./data`
            """
            self.directory = Path(directory).resolve()
            self.__datasets_paths = []
            self.__prepare_directory()

        def __prepare_directory(self):
            """
            Crea el directorio en que se almacenarán los datasets y establece
            el directorio de la caché de `kagglehub`.
            """
            self.directory.mkdir(parents=True, exist_ok=True)
            os.environ["KAGGLEHUB_CACHE"] = str(self.directory)
            print(
                f"- Directorio que contiene los datasets en CSV: {self.directory}")

        def __download_and_move_csv(
                self,
                dataset_handle: str,
                dataset_label: str
        ) -> Path:
            print(f"- Descargando '{dataset_handle}'...")

            try:
                downloaded_path = kagglehub.dataset_download(dataset_handle)

                csv_files = list(Path(downloaded_path).rglob('*.csv'))

                if csv_files:
                    if len(csv_files) > 1:
                        for source_csv in csv_files:
                            if "train" in source_csv.name:
                                destination_csv = self.directory / \
                                    (dataset_label + ".csv")

                                print(
                                    f"- Moviendo '{source_csv.name}' a '{destination_csv}'...")
                                shutil.move(str(source_csv), str(destination_csv))
                                return destination_csv

                    else:
                        source_csv = csv_files[0]
                        destination_csv = self.directory / (dataset_label + ".csv")
                        print(
                            f"- Moviendo '{source_csv.name}' a '{destination_csv}'...")
                        shutil.move(str(source_csv), str(destination_csv))
                        return destination_csv

                else:
                    print(
                        f"- No se encontró ningún archivo CSV en {downloaded_path} para {dataset_handle}.")

            except Exception as e:
                print(f"- Error al descargar o procesar {dataset_handle}: {e}")

        def __cleanup_kagglehub_cache(self):
            """
            Elimina la caché de kagglehub.
            """
            cleanup_path = self.directory / "datasets"
            if cleanup_path.exists() and cleanup_path.is_dir():
                print(
                    f"- Limpiando directorio de caché de kagglehub: {cleanup_path}")
                try:
                    shutil.rmtree(cleanup_path)
                    print("- Limpieza completada.")
                except OSError as e:
                    print(f"- Error al limpiar el directorio {cleanup_path}: {e}")

        def download_lyrics_datasets(
                self,
                df1: bool = True,
                df2: bool = False
        ) -> list[Path]:
            """
            Descarga los *datasets* utilizados en el proyecto.
            Args:
                df1 (bool): determina si se descarga el primer *dataset* (Genius Song Lyrics). Por defecto es `True`.
                df2 (bool): determina si se descarga el segundo *dataset* (Multi-Lingual Lyrics for Genre Classification).
                            Por defecto es `True`.
            """
            if df1:
                destination_paths = self.__download_and_move_csv(
                    dataset_handle="carlosgdcj/genius-song-lyrics-with-language-information",
                    dataset_label="genius-lyrics"
                )
                self.__datasets_paths.append(destination_paths)

            if df2:
                destination_paths = self.__download_and_move_csv(
                    dataset_handle="mateibejan/multilingual-lyrics-for-genre-classification",
                    dataset_label="multi_lingual-lyrics"
                )
                self.__datasets_paths.append(destination_paths)

            self.__cleanup_kagglehub_cache()

            return self.get_datasets_paths()

        def get_datasets_paths(self) -> list[list[Path]]:
            """
            Devuelve las rutas a los *datasets* en CSV.
            Output:
                list[list[Path]]
            """
            return self.__datasets_paths.copy()

    class LyricsDataLoader:
        def __init__(
            self,
            csv_paths: list[str] | list[Path],
            usecols: list[list[str]],
            chunk_size: int = 50000,
            max_rows: int = 300000,
            max_size_mb: float = 500
        ):
            self.csv_paths = csv_paths
            self.usecols = usecols
            self.chunk_size = chunk_size
            self.max_rows = max_rows
            self.max_size_mb = max_size_mb

            self.__loaded_datasets = {}
            self.__load_datasets()

        def __load_datasets(self):
            for (p, usecols) in zip(self.csv_paths, self.usecols):
                csv_size_mb = os.path.getsize(p) / (1024 * 1024)
                if csv_size_mb > self.max_size_mb:
                    chunks = []
                    total_read = 0

                    for chunk in pd.read_csv(p, usecols=usecols, chunksize=self.chunk_size):
                        chunks.append(chunk)
                        total_read += len(chunk)

                        if total_read >= self.max_rows:
                            break

                    dataset = pd.concat(chunks, ignore_index=True)

                    if len(dataset) > self.max_rows:
                        dataset = dataset.sample(n=self.max_rows)

                else:
                    dataset = pd.read_csv(p, usecols=usecols)

                self.__loaded_datasets[str(p)] = dataset

        def get_loaded_datasets(self) -> dict[str: pd.DataFrame]:
            return self.__loaded_datasets.copy()

    class LyricsDataProcessor:
        def __init__(
                self,
                output_dir: str | Path,
                datasets: list[pd.DataFrame],
                dataset_configs: list[LyricsDatasetConfig],
                genre_map: dict[str, str],
                label2id: dict[str: int],
                id2label: dict[int: str],
                clean_lyrics_fns: list[Callable[[str], str]] | None = None,
                eval_split: float | None = 0.15,
                test_split: float | None = 0.15,
                transformer_model_name: str = "roberta-base",
                device: str | torch.device = "cpu"
        ):
            """
            Constructor de la clase LyricsDataProcessor.
            Args:
                output_dir (str | Path): Directorio donde se guardarán los datos procesados.
                datasets (list[pd.DataFrame]): Lista de DataFrames con los datasets a procesar
                dataset_configs (list[LyricsDatasetConfig]): Configuraciones de los datasets.
                genre_map (dict[str, str]): Mapeo de géneros musicales.
                label2id (dict[str: int]): Mapeo de etiquetas a IDs.
                id2label (dict[int: str]): Mapeo de IDs a etiquetas.
                clean_lyrics_fns (list[Callable[[str], str]] | None): Lista de funciones para limpiar los lyrics de cada dataset.
                eval_split (float | None): Proporción del dataset para validación. Si es None, no se crea un conjunto de validación.
                test_split (float | None): Proporción del dataset para test. Si es None, no se crea un conjunto de test.
                transformer_model_name (str): Nombre del modelo transformer para tokenización.
                device (str | torch.device): Dispositivo donde se ejecutarán las operaciones (cpu o cuda).
            """
            self.output_dir = Path(output_dir)
            self.__datasets = datasets.copy()
            self.dataset_configs = dataset_configs
            self.genre_map = genre_map
            self.label2id = label2id
            self.id2label = id2label
            self.clean_lyrics_fns = clean_lyrics_fns if clean_lyrics_fns is not None else [
                None for _ in range(len(self.__datasets))]
            self.eval_split = eval_split
            self.test_split = test_split
            self.vectorizer = TfidfVectorizer(
                max_features=20000,
                ngram_range=(1, 2),
                stop_words=self.dataset_configs[0].vectorizer_config["stop_words"],
                sublinear_tf=True
            )
            self.transformer_tokenizer = AutoTokenizer.from_pretrained(
                transformer_model_name)
            self.device = device

            self.transformer_tokenize_fn = (lambda examples: self.transformer_tokenizer(
                examples["text"], padding="max_length", truncation=True, max_length=128))

            self.__X_train_raw = None
            self.__X_train_baseline = None
            self.__train_dataset_transformer = None

            self.__X_eval_raw = None
            self.__X_eval_baseline = None
            self.__eval_dataset_transformer = None

            self.__X_test_raw = None
            self.__X_test_baseline = None
            self.__test_dataset_transformer = None

            self.__y_train_raw = None
            self.__y_eval_raw = None
            self.__y_test_raw = None

        def __rename_columns(self):
            """
            Renombra las columnas de los datasets según la configuración de cada dataset.
            """
            cols_maps = [config.cols_map for config in self.dataset_configs]
            for dataset, cols_map in zip(self.__datasets, cols_maps):
                dataset.rename(columns=cols_map, inplace=True)
            print("    - Renombrado de columnas realizado.")

        def __normalize_label(self):
            """
            Normaliza la columna "label" de los datasets, convirtiendo a minúsculas y eliminando espacios.
            """
            for dataset in self.__datasets:
                dataset["label"] = (
                    dataset["label"]
                    .astype(str)
                    .str.lower()
                    .str.strip()
                )

            print("    - Normalización de label realizada.")

        def __filter_language(self):
            """
            Filtra los datasets por el idioma establecido en la configuración.
            """
            filtered = []
            for i, dataset in enumerate(self.__datasets):
                config = self.dataset_configs[i]
                language_col = config.language_col
                target_language = config.target_language
                if language_col and target_language:
                    samples_before = len(dataset)
                    dataset = dataset[dataset[language_col]
                                    == target_language].copy()
                    dataset.drop(columns=[language_col], inplace=True)
                    samples_after = len(dataset)
                    print(
                        f"    - Filtrado por idioma realizado. Antes --> {samples_before} muestras; Después --> {samples_after} muestras.")
                filtered.append(dataset)
            self.__datasets = filtered

        def __clean_genius(self, text):
            """
            Limpieza para el dataset 'Genius Song Lyrics'.
            """
            if not type(text) == str:
                return ""

            # Convertimos el texto a minúsculas
            text = text.lower()

            # Eliminamos las etiquetas [Chorus], [Intro], etc.
            text = re.sub(r"\[.*?\]", " ", text)

            # Convertimos los saltos de línea a '\n'
            text = text.replace("\r\n", '\n').replace('\r', '\n')

            # Eliminamos caracteres no alfanuméricos (excepto apostrofes y saltos)
            text = re.sub(r"[^a-z0-9\n\s']", " ", text)

            # Eliminación de whitespaces seguidos
            text = re.sub(r"[ ]{2,}", " ", text)

            # Eliminación de saltos de línea seguidos
            text = re.sub(r"\n{3,}", "\n\n", text)

            return text.strip()

        def __clean_multilingual(self, text):
            """
            Limpieza para el dataset 'Multi-Lingual Lyrics for Genre Classification'.
            """
            if not type(text) == str:
                return ""

            # Convertimos el texto a minúsculas
            text = text.lower()

            # Convertimos los saltos de línea a '\n'
            text = text.replace("\r\n", '\n').replace('\r', '\n')

            # Eliminación de whitespaces seguidos
            text = re.sub(r"[ ]{2,}", " ", text)

            # Eliminación de saltos de línea seguidos
            text = re.sub(r"\n{3,}", "\n\n", text)

            return text.strip()

        def __clean_lyrics(self):
            """
            Limpia los lyrics de los datasets aplicando las funciones de limpieza correspondientes.
            """
            for i, dataset in enumerate(self.__datasets):
                config = self.dataset_configs[i]
                name = config.name
                clean_lyrics_fn = self.clean_lyrics_fns[i]
                if name == "Genius Song Lyrics":
                    dataset["text"] = dataset["text"].apply(
                        self.__clean_genius)
                elif name == "Multi-Lingual Lyrics for Genre Classification":
                    dataset["text"].apply(
                        self.__clean_multilingual)
                else:
                    if not self.clean_lyrics_fns:
                        raise RuntimeError(
                            "- No se ha especificado ninguna función para limpiar los lyrics.")
                    dataset["text"] = dataset["text"].apply(
                        clean_lyrics_fn)

            print(f"    - Limpieza de lyrics realizada.")

        def __remove_na(self):
            """
            Elimina las muestras con valores nulos en los datasets.
            """
            samples_before = [len(dataset) for dataset in self.__datasets]
            self.__datasets = [dataset.dropna() for dataset in self.__datasets]
            samples_after = [len(dataset) for dataset in self.__datasets]
            for (sb, sa) in zip(samples_before, samples_after):
                print(
                    f"    - Eliminación de muestras con atributos nulos realizada. Antes --> {sb} muestras; Después --> {sa} muestras.")

        def __map_genres(self):
            """
            Mapea los géneros a sus identificadores numéricos en los datasets.
            """
            for dataset in self.__datasets:
                dataset["label"] = dataset["label"].map(
                    self.genre_map)
                dataset["label"] = dataset["label"].map(self.label2id)
                dataset.dropna(subset=["label"], inplace=True)
            print("    - Mapeo de géneros realizado.")

        def __unify_and_deduplicate_datasets(self):
            """
            Unifica los *datasets* y elimina muestras duplicadas.
            """
            self.__unified = pd.concat(self.__datasets)
            del self.__datasets
            self.__unified.drop_duplicates(
                subset=["text"], keep="first", inplace=True)

        def __handle_imbalance(self):
            """
            Maneja el desbalance de clases en el *dataset* unificado.
            """
            min_class_count = self.__unified["label"].value_counts().min()
            balanced_dfs = []
            for label in self.__unified["label"].unique():
                class_df = self.__unified[self.__unified["label"] == label]
                resampled = resample(class_df, n_samples=min_class_count)
                balanced_dfs.append(resampled)

            self.__unified = pd.concat(balanced_dfs, ignore_index=True)
            self.__unified = self.__unified.sample(frac=1).reset_index(drop=True)

            self.__X = self.__unified["text"]
            self.__y = self.__unified["label"].astype(int)

            del self.__unified

        def __split_dataset(self):
            """
            Divide el *dataset* en conjuntos de entrenamiento, evaluación y test según las proporciones especificadas.
            """
            if self.eval_split is not None and self.test_split is None:
                self.__X_test_raw = None
                self.__y_test_raw = None

                self.__X_train_raw, self.__X_eval_raw, self.__y_train_raw, self.__y_eval_raw = train_test_split(
                    self.__X, self.__y, test_size=self.eval_split, stratify=self.__y)
                del self.__X
                del self.__y

            elif self.eval_split is not None and self.test_split is not None:
                self.__X_train_raw, X_temp, self.__y_train_raw, y_temp = train_test_split(
                    self.__X, self.__y, test_size=(self.eval_split + self.test_split), stratify=self.__y)

                self.__X_eval_raw, self.__X_test_raw, self.__y_eval_raw, self.__y_test_raw = train_test_split(
                    X_temp, y_temp, test_size=(self.test_split / (self.eval_split + self.test_split)), stratify=y_temp
                )
                del self.__X
                del self.__y

            elif self.eval_split is None and self.test_split is not None:
                self.__X_eval_raw = None
                self.__y_eval_raw = None

                self.__X_train_raw, self.__X_test_raw, self.__y_train_raw, self.__y_test_raw = train_test_split(
                    self.__X, self.__y, test_size=self.test_split, stratify=self.__y)
                del self.__X
                del self.__y

            else:
                self.__X_eval_raw = None
                self.__y_eval_raw = None
                self.__X_test_raw = None
                self.__y_test_raw = None
                self.__X_train_raw = self.__X
                self.__y_train_raw = self.__y

        def __encode_splits(self):
            """
            Codifica los conjuntos de datos divididos utilizando el vectorizador y el tokenizador.
            """
            self.__X_train_baseline = self.vectorizer.fit_transform(
                self.__X_train_raw)

            self.__train_dataset_transformer = Dataset.from_pandas(
                pd.DataFrame({"text": self.__X_train_raw, "label": self.__y_train_raw}))
            self.__train_dataset_transformer = self.__train_dataset_transformer.map(
                self.transformer_tokenize_fn, batched=True)

            if self.__X_eval_raw is not None:
                self.__X_eval_baseline = self.vectorizer.transform(
                    self.__X_eval_raw)
                self.__eval_dataset_transformer = Dataset.from_pandas(
                    pd.DataFrame({"text": self.__X_eval_raw, "label": self.__y_eval_raw}))
                self.__eval_dataset_transformer = self.__eval_dataset_transformer.map(
                    self.transformer_tokenize_fn, batched=True)

            if self.__X_test_raw is not None:
                self.__X_test_baseline = self.vectorizer.transform(
                    self.__X_test_raw)
                self.__test_dataset_transformer = Dataset.from_pandas(pd.DataFrame(
                    {"text": self.__X_test_raw, "label": self.__y_test_raw}))
                self.__test_dataset_transformer = self.__test_dataset_transformer.map(
                    self.transformer_tokenize_fn, batched=True)

            del self.__X_train_raw
            del self.__X_eval_raw
            del self.__X_test_raw

        def harmonize_pipeline(self):
            """
            Ejecuta la pipeline completa de armonización de los datos.
            """
            print(
                f"- Ejecutando pipeline de armonización para {[config.name for config in self.dataset_configs]}...")

            self.__rename_columns()
            self.__normalize_label()
            self.__filter_language()
            self.__clean_lyrics()
            self.__remove_na()
            self.__map_genres()
            self.__unify_and_deduplicate_datasets()
            self.__handle_imbalance()
            self.__split_dataset()
            self.__encode_splits()

            print(f"- Armonización completa.")

        def get_train_data(self, return_format: str = "baseline") -> tuple[csr_matrix, np.ndarray] | Dataset:
            """
            Obtiene los datos de entrenamiento en el formato especificado.
            Args:
                return_format (str): Formato de retorno de los datos. Puede ser "baseline" o "transformer".
            Output:
                tuple[csr_matrix, np.ndarray] | Dataset
            """
            if return_format == "baseline":
                if self.__X_train_baseline is None:
                    raise RuntimeError(
                        "- No se han vectorizado los datos de entrenamiento.")

                return self.__X_train_baseline, self.__y_train_raw.values.copy()

            elif return_format == "transformer":
                if self.__train_dataset_transformer is None:
                    raise RuntimeError(
                        "- No se han tokenizado los datos de entrenamiento.")

                return self.__train_dataset_transformer

            else:
                raise RuntimeError(
                    f"- El formato 'return_format={return_format}' no está soportado. Utiliza 'baseline' o 'transformer'.")

        def get_eval_data(self, return_format: str = "baseline"):
            """
            Obtiene los datos de validación en el formato especificado.

            """
            if return_format == "baseline":
                if self.__X_eval_baseline is None:
                    raise RuntimeError(
                        "- No se han vectorizado los datos de validación.")

                return self.__X_eval_baseline, self.__y_eval_raw.values.copy()

            elif return_format == "transformer":
                if self.__eval_dataset_transformer is None:
                    raise RuntimeError(
                        "- No se han tokenizado los datos de validación.")

                return self.__eval_dataset_transformer

            else:
                raise RuntimeError(
                    f"- El formato 'return_format={return_format}' no está soportado. Utiliza 'baseline' o 'transformer'.")

        def get_test_data(self, return_format: str = "baseline"):
            if return_format == "baseline":
                if self.__X_test_baseline is None:
                    raise RuntimeError(
                        "- No se han vectorizado los datos de test.")

                return self.__X_test_baseline, self.__y_test_raw.values.copy()

            elif return_format == "transformer":
                if self.__test_dataset_transformer is None:
                    raise RuntimeError(
                        "- No se han tokenizado los datos de test.")

                return self.__test_dataset_transformer

            else:
                raise RuntimeError(
                    f"- El formato 'return_format={return_format}' no está soportado. Utiliza 'baseline' o 'transformer'.")

        def save_baseline_data(self):
            # Guardar vectorizer
            with open(self.output_dir / "vectorizer.pkl", "wb") as f:
                pickle.dump(self.vectorizer, f)

            save_npz(self.output_dir / "X_train_baseline.npz",
                    self.__X_train_baseline)
            if self.__X_eval_baseline is not None:
                save_npz(self.output_dir / "X_eval_baseline.npz",
                        self.__X_eval_baseline)
            if self.__X_test_baseline is not None:
                save_npz(self.output_dir / "X_test_baseline.npz",
                        self.__X_test_baseline)

            np.save(self.output_dir / "y_train.npy", self.__y_train_raw.values)
            if self.__y_eval_raw is not None:
                np.save(self.output_dir / "y_eval.npy", self.__y_eval_raw.values)
            if self.__y_test_raw is not None:
                np.save(self.output_dir / "y_test.npy", self.__y_test_raw.values)

            print(f"- Datos de baseline guardados en {self.output_dir}.")

        def save_transformer_data(self):
            self.__train_dataset_transformer.save_to_disk(
                self.output_dir / "train_dataset_transformer")
            self.__eval_dataset_transformer.save_to_disk(
                self.output_dir / "eval_dataset_transformer")
            self.__test_dataset_transformer.save_to_disk(
                self.output_dir / "test_dataset_transformer")

            print(f"- Datos de transformer guardados en {self.output_dir}.")

        def save_all(self):
            self.save_baseline_data()
            self.save_transformer_data()

        @staticmethod
        def load_data_baseline(path: str | Path):
            return load_npz(path)

        @staticmethod
        def load_label(path: str | Path):
            return np.load(path)

        @staticmethod
        def load_dataset_transformer(path: str | Path):
            return Dataset.load_from_disk(path)

    # src.models
    class Model(ABC):
        """
        Interfaz que sirve como base para la implementación de los modelos
        utilizados en el proyecto.
        """

        @abstractmethod
        def fit(X, y):
            pass

        @abstractmethod
        def predict(X):
            pass

    class BaselineModel(Model):
        """
        Clase que representa al modelo que se utiliza como *baseline*, que emplea
        regresión logística con SGD.
        """

        def __init__(
                self,
                epochs: int = 15,
                batch_size: int = 128
        ):
            self.epochs = epochs
            self.batch_size = batch_size

            self.clf = SGDClassifier(
                loss='log_loss',
                penalty='l2',
                max_iter=1,
                learning_rate='optimal',
                validation_fraction=0.0,
                n_jobs=-1,
                warm_start=True
            )

            self.history = {"train_acc": [], "val_acc": [],
                            'train_loss': [], 'val_loss': []}

        def fit(self, X: csr_matrix, y: np.ndarray, X_eval: csr_matrix | None = None, y_eval: np.ndarray | None = None):
            """
            Entrena el modelo utilizando mini-batch SGD.
            Args:
                X (np.ndarray): Matriz de características de entrenamiento.
                y (np.ndarray): Vector de etiquetas de entrenamiento.
                X_eval (np.ndarray | None): Matriz de características de evaluación.
                y_eval (np.ndarray | None): Vector de etiquetas de evaluación.
            """
            print(f"- Iniciando entrenamiento de {self.epochs} epochs...")

            classes = np.unique(y)

            start_time = time.time()

            for epoch in range(self.epochs):
                batch_acc = []
                for i in range(0, len(X), self.batch_size):
                    X_batch = X[i:i+self.batch_size]
                    y_batch = y[i:i+self.batch_size]

                    self.clf.partial_fit(
                        X=X_batch, y=y_batch, classes=classes)

                    batch_acc.append(self.clf.score(X_batch, y_batch))

                train_acc = np.mean(batch_acc)

                # Evaluación (sin entrenar, solo predecir)
                if X_eval is not None and y_eval is not None:
                    val_acc = self.clf.score(X_eval, y_eval)
                    self.history['val_acc'].append(val_acc)
                self.history['train_acc'].append(train_acc)

                print(
                    f"Época {epoch+1}/{self.epochs} - Train Acc: {train_acc:.4f} - Val Acc: {val_acc:.4f}")

            print(
                f"Entrenamiento finalizado en {time.time() - start_time:.2f} segundos.")

        def predict(self, X: csr_matrix) -> np.ndarray:
            """
            Realiza predicciones utilizando el modelo entrenado.
            Args:
                X (np.ndarray): Matriz de características para predecir.
            Output:
                np.ndarray: Vector de etiquetas predichas.
            """

            return self.clf.predict(X)

        def plot_training_curves(self) -> plt.Figure:
            """
            Visualiza la evolución del accuracy durante el entrenamiento.
            Output:
                plt.Figure
            """
            epochs = range(1, len(self.history['train_acc']) + 1)

            fig = plt.figure(figsize=(10, 6))
            plt.plot(epochs, self.history['train_acc'],
                    'b-o', label='Train Accuracy')
            plt.plot(epochs, self.history['val_acc'],
                    'r-o', label='Validation Accuracy')

            plt.title('Curvas de Aprendizaje: Regresión Logística (SGD)')
            plt.xlabel('Épocas')
            plt.ylabel('Accuracy')
            plt.legend()
            plt.grid(True)
            plt.show()
            return fig

        def evaluate(self, X: csr_matrix, y: np.ndarray, y_pred: np.ndarray | None = None) -> dict:
            """
            Evalúa el modelo
            Args:
                X (np.ndarray): Matriz de características.
                y (np.ndarray): Vector de etiquetas verdaderas.
            Output:
                dict
            """
            if y_pred is None:
                preds = self.predict(X)
            else:
                preds = y_pred


            acc = accuracy_score(y, preds)
            f1_macro = f1_score(y, preds, average="macro")
            f1_per_class = f1_score(y, preds, average=None)
            cm = confusion_matrix(y, preds)
            return {
                "accuracy": acc,
                "f1_macro": f1_macro,
                "f1_per_class": f1_per_class,
                "confusion_matrix": cm
            }

        def plot_confusion_matrix(self, X: csr_matrix, y: np.ndarray, y_pred: np.ndarray | None = None) -> plt.Figure:
            """
            Grafica la matriz de confusión del modelo.
            Args:
                X (np.ndarray): Matriz de características.
                y (np.ndarray): Vector de etiquetas verdaderas.
            Output:
                plt.Figure
            """
            if y_pred is None:
                preds = self.predict(X)
            else:
                preds = y_pred

            cm = confusion_matrix(y, preds)
            disp = ConfusionMatrixDisplay(confusion_matrix=cm)
            fig, ax = plt.subplots(figsize=(8, 8))
            disp.plot(ax=ax)
            plt.title('Matriz de Confusión')
            plt.show()
            return fig

        def save_model(self, path: str | Path):
            """
            Guarda el modelo en la ruta especificada.
            Args:
                path (str | Path): Ruta donde se guardará el modelo.
            """
            print(f"Guardando modelo en {path}...")
            with open(path, "wb") as f:
                pickle.dump(self.clf, f)

        @staticmethod
        def load_model(path: str | Path) -> "BaselineModel":
            """
            Carga el modelo desde la ruta especificada.
            Args:
                path (str | Path): Ruta donde está guardado el modelo.
            Output:
                BaselineModel
            """
            path = Path(path)
            print(f"- Cargando modelo desde {path}...")
            with open(path, "rb") as f:
                clf = pickle.load(f)
            model = BaselineModel()
            model.clf = clf
            return model

    class TransformerModel(Model):
        """
        Clase que representa el modelo basado en arquitectura Transformer
        que utiliza RoBERTa por defecto.
        """

        def __init__(
                self,
                output_dir: str | Path,
                label2id: dict[str: int],
                id2label: dict[int: str],
                model_name: str = 'roberta-base',
                num_labels: int = 4,
                epochs: int = 3,
                batch_train: int = 32,
                batch_eval: int = 64
        ):
            """
            Constructor de la clase TransformerModel.
            Args:
                output_dir (str | Path): Directorio donde se guardará el modelo.
                label2id (dict[str: int]): Mapeo de etiquetas a IDs.
                id2label (dict[int: str]): Mapeo de IDs a etiquetas.
                model_name (str): Nombre del modelo preentrenado de Hugging Face.
                num_labels (int): Número de etiquetas del modelo.
                epochs (int): Número de épocas para el entrenamiento.
                batch_train (int): Tamaño del batch para el entrenamiento.
                batch_eval (int): Tamaño del batch para la evaluación.
            """
            self.output_dir = output_dir
            self.label2id = label2id
            self.id2label = id2label
            self.model_name = model_name
            self.num_labels = num_labels
            self.epochs = epochs
            self.batch_train = batch_train
            self.batch_eval = batch_eval

            self._model = AutoModelForSequenceClassification.from_pretrained(
                self.model_name, num_labels=self.num_labels)

            self.dataset = None

        def compute_metrics(self, eval_pred: tuple[np.ndarray, np.ndarray]) -> dict[str, float]:
            """
            Calcula las métricas de evaluación.
            Args:
                eval_pred: Predicciones del modelo.
            Output:
                dict[str, float]
            """
            logits, labels = eval_pred
            predictions = np.argmax(logits, axis=-1)
            acc = accuracy_score(labels, predictions)
            f1 = f1_score(labels, predictions, average="macro")
            return {
                "accuracy": acc,
                "f1_macro": f1
            }

        def fit(self, train_dataset: Dataset, eval_dataset: Dataset):
            """
            Entrena el modelo con el conjunto de datos de entrenamiento y evalúa en el conjunto de validación.
            Args:
                train_dataset (Dataset): Conjunto de datos de entrenamiento.
                eval_dataset (Dataset): Conjunto de datos de validación.
            """
            print(f"- Iniciando entrenamiento del modelo {self.model_name}...")

            training_args = TrainingArguments(
                output_dir=self.output_dir,
                num_train_epochs=self.epochs,
                per_device_train_batch_size=self.batch_train,
                per_device_eval_batch_size=self.batch_eval,
                warmup_steps=500,
                weight_decay=0.01,
                logging_dir='./logs',
                logging_steps=10,
                eval_strategy="epoch",
                save_strategy="epoch",
                load_best_model_at_end=True,
                metric_for_best_model="f1_macro",
                report_to="none"
            )

            self.trainer = Trainer(
                model=self._model,
                args=training_args,
                train_dataset=train_dataset,
                eval_dataset=eval_dataset,
                compute_metrics=self.compute_metrics
            )

            self.trainer.train()
            print("Entrenamiento finalizado.")

        def predict(self, X: Dataset) -> np.ndarray:
            """
            Realiza predicciones sobre el conjunto de datos dado.
            Args:
                X (Dataset): Conjunto de datos sobre el cual se realizarán las predicciones.
            Output:
                np.ndarray
            """
            print("- Realizando predicciones...")
            predictions = self.trainer.predict(X)
            preds = np.argmax(predictions.predictions, axis=-1)
            return preds

        def plot_training_curves(self) -> plt.Figure:
            """
            Visualiza la evolución del accuracy durante el entrenamiento.
            Output:
                plt.Figure
            """
            log_history = self.trainer.state.log_history

            eval_epochs = []
            eval_acc = []

            for log in log_history:
                if "epoch" in log:
                    if "eval_accuracy" in log:
                        eval_epochs.append(log["epoch"])
                        eval_acc.append(log["eval_accuracy"])

            fig = plt.figure(figsize=(10, 6))

            plt.plot(eval_epochs, eval_acc, 'r-o',
                    label='Validation Accuracy')

            plt.xlabel('Épocas')
            plt.ylabel('Accuracy')
            plt.title('Curvas de Aprendizaje: Transformer (RoBERTa)')
            plt.legend(fontsize=11)
            plt.grid(True, alpha=0.3)

            plt.tight_layout()
            plt.show()

            return fig

        def evaluate(self, dataset: Dataset, y_pred: np.ndarray | None = None) -> dict:
            """
            Evalúa el modelo.
            Args:
                X (Dataset): Conjunto de datos para evaluar.
                y (np.ndarray): Etiquetas verdaderas.
            Output:
                dict
            """
            print("- Evaluando el modelo...")
            if y_pred is None:
                preds = self.trainer.predict(dataset)
            else:
                preds = y_pred

            acc = accuracy_score(dataset["label"], preds)
            f1_macro = f1_score(dataset["label"], preds, average='macro')
            f1_per_class = f1_score(dataset["label"], preds, average=None)
            cm = confusion_matrix(dataset["label"], preds)

            return {
                "accuracy": acc,
                "f1_macro": f1_macro,
                "f1_per_class": f1_per_class,
                "confusion_matrix": cm
            }

        def plot_confusion_matrix(self, dataset: Dataset, y_pred: np.ndarray | None = None) -> plt.Figure:
            """
            Grafica la matriz de confusión del modelo.
            Args:
                dataset (Dataset): Conjunto de datos para evaluar.
            Output:
                plt.Figure
            """
            print("- Graficando matriz de confusión...")
            if y_pred is None:
                preds = self.predict(dataset)
            else:
                preds = y_pred

            cm = confusion_matrix(dataset["label"], preds)
            disp = ConfusionMatrixDisplay(
                confusion_matrix=cm, display_labels=list(self.label2id.keys()))

            fig, ax = plt.subplots(figsize=(8, 8))
            disp.plot(ax=ax, cmap=plt.cm.Blues, colorbar=False)
            plt.title('Matriz de Confusión')
            plt.show()

            return fig

        def save_model(self, path: str | Path):
            """
            Guarda el modelo en la ruta especificada.
            Args:
                path (str | Path): Ruta donde se guardará el modelo.
            """
            print(f"- Guardando modelo en {path}...")
            self.trainer.save_model(path)

        @staticmethod
        def load_model(path: str | Path, label2id: dict | None = None, id2label: dict | None = None) -> "TransformerModel":
            """
            Carga el modelo desde la ruta especificada.
            Args:
                path: Ruta del modelo guardado
                label2id (dict): Mapeo de etiquetas a IDs.
                id2label (dict): Mapeo de IDs a etiquetas.
            Returns:
                TransformerModel
            """
            print(f"- Cargando modelo desde {path}...")
            model = AutoModelForSequenceClassification.from_pretrained(path)
            instance = TransformerModel(
                output_dir=path,
                label2id=label2id or {},
                id2label=id2label or {},
                num_labels=model.config.num_labels
            )
            instance._model = model
            instance.trainer = Trainer(model=model)
            return instance

    class ErrorAnalyzer:
        def __init__(self, model, X_test, y_test, y_pred, id2label, vectorizer=None, tokenizer=None):
            """
            Constructor de la clase ErrorAnalyzer para análisis de explicabilidad.
            Compatible con BaselineModel (SGD) y TransformerModel.

            Args:
                model: Modelo entrenado (BaselineModel o TransformerModel)
                X_test: Matriz de características de test (vectorizada o Dataset)
                y_test: Etiquetas reales de test
                y_pred: Predicciones del modelo
                id2label: Diccionario {id: etiqueta_texto}
                vectorizer: TfidfVectorizer (para modelos baseline)
                tokenizer: Tokenizer de Hugging Face (para modelos transformer)
            """
            self.model = model
            self.X_test = X_test
            self.y_test = np.array(y_test) if isinstance(
                y_test, (pd.Series, list)) else y_test
            self.y_pred = np.array(y_pred) if isinstance(
                y_pred, (pd.Series, list)) else y_pred
            self.id2label = id2label
            self.vectorizer = vectorizer
            self.tokenizer = tokenizer

            # Detectar tipo de modelo
            self.model_type = self._detect_model_type()

            # Obtener nombres de features del vectorizador
            self.feature_names = None
            if self.vectorizer is not None and hasattr(self.vectorizer, 'get_feature_names_out'):
                self.feature_names = self.vectorizer.get_feature_names_out()

        def _detect_model_type(self):
            """
            Detecta si es un BaselineModel (con .clf) o TransformerModel (con .trainer).
            """
            if hasattr(self.model, 'clf'):
                return 'baseline'
            elif hasattr(self.model, 'trainer'):
                return 'transformer'
            else:
                return 'unknown'

        def _reconstruct_text_from_vectorizer(self, idx):
            """
            Reconstruye el texto desde la matriz sparse TF-IDF.
            """
            if self.vectorizer is None or self.feature_names is None:
                return "[Texto no disponible]"

            x_vec = self.X_test[idx]

            if hasattr(x_vec, 'indices'):
                feature_indices = x_vec.indices
                feature_values = x_vec.data
            else:
                feature_indices = np.where(x_vec > 0)[0]
                feature_values = x_vec[feature_indices]

            words_with_scores = [(self.feature_names[idx], feature_values[i])
                                for i, idx in enumerate(feature_indices)]
            words_with_scores.sort(key=lambda x: x[1], reverse=True)

            top_words = [word for word, score in words_with_scores[:100]]
            return " ".join(top_words)

        def _reconstruct_text_from_tokenizer(self, idx):
            """
            Reconstruye el texto desde los token IDs usando el tokenizador.
            """
            if self.tokenizer is None:
                return "[Texto no disponible]"

            try:
                # Ensure idx is a standard Python int
                idx_int = int(idx)
                if hasattr(self.X_test, '__getitem__'):
                    example = self.X_test[idx_int] # Changed to idx_int

                    if isinstance(example, dict):
                        if 'input_ids' in example:
                            token_ids = example['input_ids']
                        else:
                            token_ids = list(example.values())[0]
                    else:
                        return "[Formato desconocido]"

                    text = self.tokenizer.decode(
                        token_ids, skip_special_tokens=True)
                    return text
            except Exception as e:
                return f"[Error: {str(e)}]"

            return "[Texto no disponible]"

        def get_text(self, idx):
            """Obtiene el texto reconstruido."""
            if self.tokenizer is not None:
                return self._reconstruct_text_from_tokenizer(idx)
            elif self.vectorizer is not None:
                return self._reconstruct_text_from_vectorizer(idx)
            return "[Texto no disponible]"

        def get_misclassified_examples(self, n=10):
            """
            Retorna los n ejemplos mal clasificados con más detalles.
            """
            errors_mask = self.y_test != self.y_pred
            error_indices = np.where(errors_mask)[0]

            if len(error_indices) == 0:
                return pd.DataFrame()

            selected_indices = np.random.choice(
                error_indices, min(n, len(error_indices)), replace=False)

            results = []
            for idx in selected_indices:
                real_id = self.y_test[idx]
                pred_id = self.y_pred[idx]
                text = self.get_text(idx)
                text_snippet = text[:250] + "..." if len(text) > 250 else text

                results.append({
                    'índice': idx,
                    'texto_muestra': text_snippet,
                    'etiqueta_real': self.id2label[real_id],
                    'etiqueta_predicha': self.id2label[pred_id],
                })

            return pd.DataFrame(results)

        def explain_prediction(self, text_idx, top_n=10):
            """
            Explica una predicción mostrando las palabras más influyentes.
            Compatible con ambos tipos de modelos.
            """
            pred_id = self.y_pred[text_idx]
            real_id = self.y_test[text_idx]

            print(f"- EXPLICACIÓN DE PREDICCIÓN - Ejemplo #{text_idx}")
            print(f"    - Predicción: {self.id2label[pred_id]}")
            print(f"    - Etiqueta Real: {self.id2label[real_id]}")
            print(f"    - Predicción acertada: {"Sí" if pred_id == real_id else "No"} ")

            # Mostrar texto reconstruido
            text = self.get_text(text_idx)
            print(f"    - Texto:\n{text[:500]}...\n")

            # Explicabilidad según tipo de modelo
            if self.model_type == 'baseline':
                self._explain_baseline(text_idx, pred_id, real_id, top_n)
            else:
                print("Modelo no soportado para explicabilidad")

            print(f"{'='*80}\n")

        def _explain_baseline(self, text_idx, pred_id, real_id, top_n):
            """
            Explicabilidad para BaselineModel (accede a model.clf.coef_).
            """
            x_vec = self.X_test[text_idx]

            if not hasattr(self.model.clf, 'coef_') or self.feature_names is None:
                print("- BaselineModel no tiene coeficientes disponibles")
                return

            if hasattr(x_vec, 'indices'):
                feature_indices = x_vec.indices
                feature_values = x_vec.data
            else:
                feature_indices = np.where(x_vec > 0)[0]
                feature_values = x_vec[feature_indices]

            self._print_top_features_explanation(
                self.model.clf.coef_, pred_id, feature_indices, feature_values,
                f"Palabras influyentes para '{self.id2label[pred_id]}'", top_n)

            if pred_id != real_id:
                self._print_top_features_explanation(
                    self.model.clf.coef_, real_id, feature_indices, feature_values,
                    f"Palabras influyentes para '{self.id2label[real_id]}'", top_n)

        def _print_top_features_explanation(self, coef, class_id, feature_indices, feature_values, title, top_n):
            """
            Imprime las features más importantes para una clase.
            """
            class_coefs = coef[class_id]
            contributions = class_coefs[feature_indices] * feature_values
            sorted_idx = np.argsort(contributions)[::-1]

            print(f"- {title}:")
            print(
                f"    {'Palabra':<25} {'Score TF-IDF':<15} {'Coeficiente':<15} {'Influencia':<15}")

            count = 0
            for i in sorted_idx:
                if count >= top_n or contributions[i] <= 0:
                    continue

                feat_idx = feature_indices[i]
                word = self.feature_names[feat_idx]
                tfidf = feature_values[i]
                coef_val = class_coefs[feat_idx]
                contrib = contributions[i]

                print(f"    {word:<25} {tfidf:<15.6f} {coef_val:<15.6f} {contrib:<15.6f}")
                count += 1

        def analyze_error_patterns(self):
            """
            Analiza patrones en los errores cometidos por el modelo.
            """
            print("- ANÁLISIS DE PATRONES DE ERROR")

            errors_mask = self.y_test != self.y_pred
            total_errors = errors_mask.sum()
            total_samples = len(self.y_test)
            error_rate = total_errors / total_samples * 100

            print(f"    - Errores totales: {total_errors}/{total_samples} ({error_rate:.2f}%)")

            error_pairs = {}
            for i in np.where(errors_mask)[0]:
                real = self.id2label[self.y_test[i]]
                pred = self.id2label[self.y_pred[i]]
                key = f"{real} → {pred}"
                error_pairs[key] = error_pairs.get(key, 0) + 1

            for pair in sorted(error_pairs.items(), key=lambda x: x[1], reverse=True):
                print(
                    f"        - {pair[0]:<30} {pair[1]:<10} {(pair[1]/total_errors)*100:.1f}%")


        def analyze_subgroups(self):
            """
            Analiza el rendimiento según la longitud del texto reconstruido.
            """
            print("- ANÁLISIS DE RENDIMIENTO POR LONGITUD DE TEXTO")

            # Calcular longitud de textos (limitado para eficiencia con transformers)
            text_lengths = []
            limit = min(len(self.y_test), 1000)
            for idx in range(limit):
                text = self.get_text(idx)
                text_lengths.append(len(text.split()))

            # Crear DataFrame
            df_analysis = pd.DataFrame({
                'length': text_lengths,
                'correct': (self.y_test[:len(text_lengths)] == self.y_pred[:len(text_lengths)])
            })

            # Definir rangos
            bins = [0, 20, 50, 100, 200, 500, 10000]
            labels = ['Muy Corto', 'Corto', 'Medio',
                    'Largo', 'Muy Largo', 'Extremadamente Largo']

            df_analysis['length_group'] = pd.cut(
                df_analysis['length'], bins=bins, labels=labels)

            # Agrupar y mostrar
            grouped = df_analysis.groupby('length_group', observed=True)['correct'].agg(
                ['count', 'sum', 'mean'])
            grouped.columns = ['Total', 'Correctas', 'Accuracy']
            grouped['Accuracy %'] = (
                grouped['Accuracy'] * 100).apply(lambda x: f"{x:.1f}%")

            print(grouped[['Total', 'Correctas', 'Accuracy %']])

            # Visualizar
            fig, ax = plt.subplots(figsize=(12, 6))
            accuracy_values = grouped['Accuracy'].values
            grouped_labels = grouped.index.astype(str)
            colors = ['green' if x > 0.7 else 'orange' if x >
                    0.5 else 'red' for x in accuracy_values]

            ax.bar(range(len(grouped_labels)),
                accuracy_values, color=colors)
            ax.set_xticks(range(len(grouped_labels)))
            ax.set_xticklabels(grouped_labels, rotation=45, ha='right')
            ax.set_title('Accuracy por Longitud del Texto')
            ax.set_xlabel('Rango de Longitud (palabras)')
            ax.set_ylabel('Accuracy')
            ax.grid(axis='y')
            ax.set_ylim([0, 1.05])

            plt.tight_layout()
            plt.show()

            return fig


else:
    from src.utils.lyrics_dataset_manager import LyricsDatasetManager
    from src.utils.lyrics_dataset_config import LyricsDatasetConfig
    from src.utils.lyrics_data_loader import LyricsDataLoader
    from src.utils.lyrics_data_processor import LyricsDataProcessor
    from src.models.baseline_model import BaselineModel
    from src.models.transformer_model import TransformerModel
    from src.analysis.error_analysis import ErrorAnalyzer

In [ ]:
print("- Versiones de las librerías utilizadas:")
print(f"    - NumPy: {np.__version__}")
print(f"    - Pandas: {pd.__version__}")
print(f"    - scikit-learn: {sklearn.__version__}")
print(f"    - PyTorch: {torch.__version__}")
print(f"    - Matplotlib: {matplotlib.__version__}")

In [ ]:
# Dispositivo para PyTorch
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Semilla
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.random.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
sklearn.random.seed(SEED)

# ¿Descargar datasets?
DOWNLOAD_DATASETS = False

# Directorios
DATA_DIR = Path("data")
MODELS_DIR = Path("models")
TRANSFORMER_OUTPUT_DIR = Path(MODELS_DIR / "transformer_output")

# ¿Datasets preprocesados?
USE_PREPROCESSED_DATASETS = True
PREPROCESSED_DATA_DIR = Path(DATA_DIR / "generated")

# ¿Entrenar?
TRAIN_MODEL = False

# Splits del dataset
EVAL_SPLIT = 0.15
TEST_SPLIT = 0.15

# Mapeo de géneros (labels)
GENRE_MAP = {
    # Géneros principales
    "rap": "rap",
    "hip-hop": "rap",
    "rock": "rock",
    "pop": "pop",

    # Otros géneros
    "rb": "other",
    "r&b": "other",
    "metal": "other",
    "indie": "other",
    "folk": "other",
    "jazz": "other",
    "electronic": "other",
    "edm": "other",
    "country": "other",
    "misc": "other",
    "soul": "other",
    "blues": "other",
    "classical": "other",
}
LABEL2ID = {
    "rap": 0,
    "rock": 1,
    "pop": 2,
    "other": 3
}
ID2LABEL = {
    0: "rap",
    1: "rock",
    2: "pop",
    3: "other"
}

# Configuración de los datasets
GENIUS_CONFIG_PATH = Path("config/genius-song-lyrics.json")
MULTILINGUAL_CONFIG_PATH = Path("config/multi_lingual-lyrics-for-genre-classification.json")


# Modelo basado en Transformer
TRANSFORMER_MODEL_NAME = "roberta-base"

# Mostrar parámetros
print(f"- Dispositivo utilizado por PyTorch: {DEVICE}")
print(f"- Semilla utilizada para reproducibilidad: {SEED}")
print(f"- Descarga de datasets: {'Sí' if DOWNLOAD_DATASETS else 'No'}")

# Procesamiento del Lenguaje Natural 2 - Práctica 1

## 1. Obtención y armonización de *datasets*
---
### 1.1. Obtenemos los *datasets* de **Kaggle** utilizando `kagglehub` en la clase `LyricsDatasetManager`
- Mediante la clase `LyricsDatasetManager` se descargan los datasets "Genius Song Lyrics" y "Multi-Lingual Lyrics for Genre Classification" de Kaggle.

In [ ]:
if not USE_PREPROCESSED_DATASETS:
    if DOWNLOAD_DATASETS:
        dataset_manager = LyricsDatasetManager(directory=DATA_DIR)
        dataset_manager.download_lyrics_datasets(df1=True, df2=True)
        dataset_paths = dataset_manager.get_datasets_paths()

    else:
        dataset_paths = [DATA_DIR/"genius-lyrics.csv", DATA_DIR/"multi_lingual-lyrics.csv"]

### 1.2. Cargamos los datos desde los ficheros CSV utilizando la clase `LyricsDataLoader`

In [ ]:
# Obtenemos las configuraciones de los datasets
if not USE_PREPROCESSED_DATASETS:
    genius_config = LyricsDatasetConfig(config_path=GENIUS_CONFIG_PATH)
    multilingual_config = LyricsDatasetConfig(config_path=MULTILINGUAL_CONFIG_PATH)

In [ ]:
if not USE_PREPROCESSED_DATASETS:
    # Cargamos los datasets
    data_loader = LyricsDataLoader(
        csv_paths=dataset_paths,
        usecols=[genius_config.cols_map.keys(), multilingual_config.cols_map.keys()]
    )
    datasets = list(data_loader.get_loaded_datasets().values())

### 1.3. Preprocesamos y armonizamos los datos cargados mediante la clase `LyricsDataProcessor`

In [ ]:
if not USE_PREPROCESSED_DATASETS:
    # Preprocesamos el dataset
    data_processor = LyricsDataProcessor(
        datasets=datasets,
        dataset_configs=[genius_config, multilingual_config],
        genre_map=GENRE_MAP,
        label2id=LABEL2ID,
        id2label=ID2LABEL,
        eval_split=EVAL_SPLIT,
        test_split=TEST_SPLIT,
        transformer_model_name=TRANSFORMER_MODEL_NAME,
        device=DEVICE
    )
    data_processor.harmonize_pipeline()

#### 1.3.1. Explicación de los parámetros usados en la vectorización:

1. `max_features=20000 (Control de Dimensionalidad)`
    - El problema: El vocabulario en un corpus de 500.000 canciones es inmenso. Si cuentas todas las palabras únicas (incluyendo erratas o nombres propios raros), podrías tener 200.000 o 300.000 columnas. Esto haría que el entrenamiento fuera lentísimo y consumiría toda tu memoria RAM.
    - La solución: Le decimos al modelo: "Quédate solo con las 20.000 palabras más frecuentes e importantes; ignora el resto".
    - Por qué 20k: Es un sweet spot (punto dulce). Captura suficiente vocabulario para diferenciar géneros sin incluir el "ruido" de palabras que solo aparecen una vez en todo el dataset (que suelen llevar a sobreajuste).
    
2. `ngram_range=(1, 2) (Contexto)`
    - El problema: Si usamos solo palabras sueltas (unigramas), perdemos el contexto.
    - Ejemplo: La palabra "Heavy" por sí sola no dice mucho.La palabra "Metal" por sí sola puede ser un material.
    - La solución: Al poner (1, 2), el modelo analiza unigramas (palabras sueltas) Y bigramas (pares de palabras consecutivas).
        - Ahora el modelo ve "Heavy Metal" como una característica propia.
        - Ve "I love" o "Gangsta Rap" como unidades de significado.
    - Por qué es útil en música: Los géneros musicales se definen mucho por frases hechas y combinaciones de palabras, no solo por palabras aisladas.
    
3. `stop_words='english' (Limpieza de Ruido)`
    - El problema: Las palabras más frecuentes en inglés son determinantes y preposiciones (the, is, a, and, of). Estas aparecen tanto en canciones de Rap como de Country. No aportan información para distinguir el género.
    - La solución: Eliminarlas reduce drásticamente el tamaño de la matriz y obliga al modelo a fijarse en sustantivos, verbos y adjetivos, que son los que llevan la carga semántica (ej: "truck" vs "flow").
    - Justificación: En tu notebook pruebas.ipynb filtramos explícitamente por df["language"] == "en", por lo que sabemos seguro que el idioma es inglés.

4. `sublinear_tf=True (Suavizado de Repeticiones)`
    - El problema: Las canciones, especialmente el Pop y el Rock, son extremadamente repetitivas (estribillos).
        - Imagina una canción que dice "Baby, baby, baby, baby..." 50 veces.
        - En un conteo normal, la palabra "baby" tendría un peso de 50.
        - Otra canción que la diga 1 vez tendría un peso de 1.
        - ¿Es la primera canción "50 veces más" de género Pop que la segunda? Probablemente no. Esa diferencia lineal es exagerada y sesga al modelo.
    - La solución: sublinear_tf aplica una escala logarítmica: $1 + \log(TF)$.Si aparece 1 vez $\rightarrow$ peso 1.
        - Si aparece 50 veces $\rightarrow$ peso $\approx 2.7$.
    - Resultado: Penalizamos la repetición excesiva típica de los estribillos para que una sola palabra repetida no domine toda la predicción.

### 1.4. Obtenemos los datos de entrenamiento, validación y *test*
- En la clase `LyricsDataProcessor` se aplica una división estratificada del conjunto de datos en entrenamiento (70%), validación (15%) y *test* (15%) para mantener la proporción de géneros en cada subconjunto.
- Además, utilizando el método `get_[split]_data()` podemos obtener los datos vectorizados o tokenizados según sea necesario para los diferentes modelos.

In [ ]:
if not USE_PREPROCESSED_DATASETS:
    # Obtenemos los datos preparados para BaselineModel
    X_train_baseline, y_train_baseline = data_processor.get_train_data(return_format="baseline")
    X_eval_baseline, y_eval_baseline = data_processor.get_eval_data(return_format="baseline")
    X_test_baseline, y_test_baseline = data_processor.get_test_data(return_format="baseline")

    # Obtenemos los datos preparados para TransformerModel
    train_dataset_transformer, y_train_transformer = data_processor.get_train_data(return_format="transformer")
    eval_dataset_transformer, y_eval_transformer = data_processor.get_eval_data(return_format="transformer")
    test_dataset_transformer, y_test_transformer = data_processor.get_test_data(return_format="transformer")

else:
    X_train_baseline = LyricsDataProcessor.load_data_baseline(PREPROCESSED_DATA_DIR / "X_train_baseline.npz")
    y_train_baseline = LyricsDataProcessor.load_label(PREPROCESSED_DATA_DIR / "y_train.npy")
    X_eval_baseline = LyricsDataProcessor.load_data_baseline(PREPROCESSED_DATA_DIR / "X_eval_baseline.npz")
    y_eval_baseline = LyricsDataProcessor.load_label(PREPROCESSED_DATA_DIR / "y_eval.npy")
    X_test_baseline = LyricsDataProcessor.load_data_baseline(PREPROCESSED_DATA_DIR / "X_test_baseline.npz")
    y_test_baseline = LyricsDataProcessor.load_label(PREPROCESSED_DATA_DIR / "y_test.npy")

    train_dataset_transformer = LyricsDataProcessor.load_dataset_transformer(
        PREPROCESSED_DATA_DIR / "train_dataset_transformer")
    eval_dataset_transformer = LyricsDataProcessor.load_dataset_transformer(
        PREPROCESSED_DATA_DIR / "eval_dataset_transformer")
    test_dataset_transformer = LyricsDataProcessor.load_dataset_transformer(
        PREPROCESSED_DATA_DIR / "test_dataset_transformer")

## 2. Entrenamiento y evaluación de los modelos
### 2.1. Entrenamiento o carga de los modelos
- Si los modelos ya han sido entrenados y guardados en disco, se cargan directamente. Si no, se entrenan desde cero utilizando los datos de entrenamiento y validación obtenidos previamente.

In [ ]:
if TRAIN_MODEL:
    # Entrenamos el modelo Baseline
    baseline_model = BaselineModel(epochs=15, batch_size=256)
    baseline_model.fit(X_train_baseline, y_train_baseline,
                       X_eval_baseline, y_eval_baseline)

    # Entrenamos el modelo Transformer
    transformer_model = TransformerModel(
        output_dir=TRANSFORMER_OUTPUT_DIR,
        label2id=LABEL2ID,
        id2label=ID2LABEL)
    transformer_model.fit(train_dataset_transformer, eval_dataset_transformer)

else:
    # Cargamos el modelo Baseline
    baseline_model = BaselineModel.load_model(MODELS_DIR / "baseline_model.pkl")

    # Cargamos el modelo Transformer
    transformer_model = TransformerModel.load_model(
        MODELS_DIR / "transformer_model", label2id=LABEL2ID, id2label=ID2LABEL)

### 2.2. Evaluación de los modelos con el conjunto de *test*
- Cada modelo se evalúa utilizando el conjunto de *test* y se calculan métricas como *accuracy*, *f1-score macro* y *f1-score* para cada clase.

In [ ]:
# Obtener predicciones
baseline_y_pred = baseline_model.predict(X_test_baseline)
transformer_y_pred = transformer_model.predict(test_dataset_transformer)

In [ ]:
# Evaluamos el modelo Baseline
baseline_results = baseline_model.evaluate(X_test_baseline, y_test_baseline, baseline_y_pred)
print("- Resultados del modelo Baseline:")
print(f"    - Accuracy: {baseline_results['accuracy']:.4f}")
print(f"    - F1 Macro: {baseline_results['f1_macro']:.4f}")
for idx, f1 in enumerate(baseline_results['f1_per_class']):
    print(f"        - F1 Clase {ID2LABEL[idx]}: {f1:.4f}")

# Evaluamos el modelo Transformer
transformer_results = transformer_model.evaluate(test_dataset_transformer, transformer_y_pred)
print("- Resultados del modelo Transformer:")
print(f"    - Accuracy: {transformer_results['accuracy']:.4f}")
print(f"    - F1 Macro: {transformer_results['f1_macro']:.4f}")
for idx, f1 in enumerate(transformer_results['f1_per_class']):
    print(f"        - F1 Clase {ID2LABEL[idx]}: {f1:.4f}")

### 2.3. Matriz de confusión con el conjunto de *test*
- Se genera y visualiza la matriz de confusión para cada modelo utilizando el conjunto de *test* para analizar los errores de clasificación.

In [ ]:
cm_fig_baseline = baseline_model.plot_confusion_matrix(X_test_baseline, y_test_baseline, baseline_y_pred)

In [ ]:
cm_fig_transformer = transformer_model.plot_confusion_matrix(test_dataset_transformer, transformer_y_pred)

## 3. Explicabilidad y Análisis de Errores utilizando la clase `ErrorAnalyzer`

In [ ]:
# Utilizamos el mismo vectorizador y tokenizador que en la clase LyricsDataProcessor
vectorizer = TfidfVectorizer(
    max_features=20000,
    ngram_range=(1, 2),
    stop_words="english",
    sublinear_tf=True
)

tokenizer = AutoTokenizer.from_pretrained("roberta-base")

In [ ]:
# Inicializar Analizador
with open(PREPROCESSED_DATA_DIR / "vectorizer.pkl", "rb") as f:
    vectorizer = pickle.load(f)

baseline_analyzer = ErrorAnalyzer(
    model=baseline_model,
    X_test=X_test_baseline,
    y_test=y_test_baseline,
    y_pred=baseline_y_pred,
    id2label=ID2LABEL,
    vectorizer=vectorizer
)

transformer_analyzer = ErrorAnalyzer(
    model=transformer_model,
    X_test=test_dataset_transformer,
    y_test=np.array(test_dataset_transformer["label"]),
    y_pred=transformer_y_pred,
    id2label=ID2LABEL,
    tokenizer=tokenizer
)

### 3.1. Ejemplos mal clasificados

In [ ]:
baseline_errors_df = baseline_analyzer.get_misclassified_examples(n=10)
print(baseline_errors_df)

In [ ]:
# Re-initialize transformer_analyzer with correct y_test type
transformer_errors_df = transformer_analyzer.get_misclassified_examples(n=10)
print(transformer_errors_df)

### 3.2. Explicabilidad (coeficientes del Modelo)
- Analizamos qué palabras contribuyeron más a la predicción.

In [ ]:
# Explicar el primer error encontrado (si existe)
if baseline_errors_df is not None and not baseline_errors_df.empty:
    idx_to_explain = baseline_errors_df.iloc[0]['índice']
    baseline_analyzer.explain_prediction(idx_to_explain)
else:
    print("No hay errores para explicar, mostrando un ejemplo correcto aleatorio.")
    baseline_analyzer.explain_prediction(0)

In [ ]:
if transformer_errors_df is not None and not transformer_errors_df.empty:
    idx_to_explain = transformer_errors_df.iloc[0]['índice']
    transformer_analyzer.explain_prediction(idx_to_explain)
else:
    print("No hay errores para explicar, mostrando un ejemplo correcto aleatorio.")
    transformer_analyzer.explain_prediction(0)

### 3.3. Análisis de subgrupos (longitud del texto)

In [ ]:
baseline_subgroups_fig = baseline_analyzer.analyze_subgroups()

In [ ]:
transformer_subgroups_fig = transformer_analyzer.analyze_subgroups()